In [2]:
from pymilvus import AsyncMilvusClient

client = AsyncMilvusClient(url="http://127.0.0.1:19530")

In [5]:
await client.list_databases()

['default', 'deal_pilot']

In [4]:
db_name = "deal_pilot"

if db_name not in await client.list_databases():
    await client.create_database(db_name)

In [6]:
await client.use_database(db_name)

In [8]:
from pymilvus import DataType

schema = client.create_schema(
    auto_id=True,
    enable_dynamic_field=True
)

schema.add_field(field_name="id",datatype=DataType.INT64,is_primary=True)

schema.add_field(field_name="text",datatype=DataType.VARCHAR,max_length=5000)

schema.add_field(field_name="vector",datatype=DataType.FLOAT_VECTOR,dim=1024)

{'auto_id': True, 'description': '', 'fields': [{'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'is_primary': True, 'auto_id': True}, {'name': 'text', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 5000}}, {'name': 'vector', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 1024}}], 'enable_dynamic_field': True, 'enable_namespace': False}

In [9]:
collection_name = "feishu_doc"

if collection_name not in await client.list_collections():
    await client.create_collection(collection_name=collection_name,schema=schema)

In [14]:
index_params = client.prepare_index_params()

index_name = "vector_index"

index_params.add_index(
    field_name="vector",
    index_name=index_name,
    index_type="HNSW",
    metric_type="COSINE",
)

await client.create_index(
    collection_name=collection_name,
    index_params=index_params,
    sync = False
)

In [11]:
await client.list_indexes(collection_name=collection_name)

['vector_index']

In [12]:
await client.describe_index(
    collection_name=collection_name,
    index_name=index_name
)

{'metric_type': 'COSINE',
 'index_type': 'HNSW',
 'field_name': 'vector',
 'index_name': 'vector_index',
 'total_rows': 0,
 'indexed_rows': 0,
 'pending_index_rows': 0,
 'state': 'Finished'}